# Regresión Lineal - Accidentes de tránsito en Bogotá

## Notebook 2: Agregación diaria y construcción de features

Continúa desde `01_exploracion_y_limpieza.ipynb`. Parte del dataset ya limpio (`data/processed/accidentes_limpios.csv`) y construye la serie diaria de accidentes por localidad.

In [1]:
import pandas as pd
from pathlib import Path

ruta_datos_limpios = Path.cwd().parent / "data" / "processed" / "accidentes_limpios.csv"

df_clean = pd.read_csv(ruta_datos_limpios, parse_dates=["FECHA_HORA_ACC"])
df_clean.shape

(199093, 16)

## 1. Agregación diaria por localidad

Agrupamos por fecha (sin hora) y localidad, contando cuántos accidentes hubo en cada combinación.

In [2]:
group_accidents_by_date_and_location = (
    df_clean.groupby([df_clean['FECHA_HORA_ACC'].dt.date, 'LOCALIDAD'])
            .size()
            .reset_index(name='num_accidentes')
            .rename(columns={'FECHA_HORA_ACC': 'fecha'})
)

group_accidents_by_date_and_location.shape

(41229, 3)

In [3]:
group_accidents_by_date_and_location.head(10)

,fecha,LOCALIDAD,num_accidentes
0,2015-01-01,BOSA,1
1,2015-01-01,CHAPINERO,1
2,2015-01-01,CIUDAD BOLIVAR,3
3,2015-01-01,ENGATIVA,2
4,2015-01-01,FONTIBON,1
5,2015-01-01,KENNEDY,1
6,2015-01-01,PUENTE ARANDA,1
7,2015-01-01,SUBA,2
8,2015-01-02,BARRIOS UNIDOS,1
9,2015-01-02,BOSA,2


## 2. Completar combinaciones fecha-localidad

La agregación anterior solo tiene filas para combinaciones donde ocurrió al menos un accidente. Para una serie temporal completa, generamos todas las combinaciones posibles de fecha x localidad y rellenamos con 0 los días sin accidentes.

In [4]:
all_dates = pd.date_range(
    start=group_accidents_by_date_and_location['fecha'].min(),
    end=group_accidents_by_date_and_location['fecha'].max(),
    freq='D'
)

unique_locations = df_clean['LOCALIDAD'].unique()

all_combinations = pd.MultiIndex.from_product(
    [all_dates, unique_locations],
    names=['fecha', 'LOCALIDAD']
)

group_accidents_by_date_and_location['fecha'] = pd.to_datetime(group_accidents_by_date_and_location['fecha'])

all_registers = (
    group_accidents_by_date_and_location.set_index(['fecha', 'LOCALIDAD'])
                     .reindex(all_combinations, fill_value=0)
                     .reset_index()
)

all_registers.shape

(46455, 3)

In [5]:
all_registers.head(10)

,fecha,LOCALIDAD,num_accidentes
0,2015-01-01,ENGATIVA,2
1,2015-01-01,PUENTE ARANDA,1
2,2015-01-01,USAQUEN,0
3,2015-01-01,CIUDAD BOLIVAR,3
4,2015-01-01,LOS MARTIRES,0
5,2015-01-01,SUBA,2
6,2015-01-01,FONTIBON,1
7,2015-01-01,USME,0
8,2015-01-01,TEUSAQUILLO,0
9,2015-01-01,BARRIOS UNIDOS,0
